# D1: эксперимент по выбору лёгкого классификатора intent для AI-ядра

## Контекст эксперимента

Этот ноутбук — рабочая среда для эмпирического сравнения **5 канонических
baseline'ов** закрытой классификации intent пользовательского сообщения
(`anamnesis`, `faq`, `booking`, `unsupported`). Цель — выбрать архитектуру,
которая в production AI-ядра заменит LLM-роутер с сохранением медицинской
безопасности.

## Что сравниваем

Закрытая классификация без режима отказа: модель обязана вернуть один из
четырёх классов для каждого сообщения. Никаких порогов, defer, селективных
политик и каскадов — только честное сравнение архитектур.

| ID | Модель | Архитектура |
|---|---|---|
| B0 | `B0_rules` | regex / keywords (нижняя граница без обучения) |
| B1.1 | `B1.1_tfidf_lr` | TF-IDF char_wb(2,5) + Logistic Regression |
| B1.3 | `B1.3_fasttext` | fastText supervised (hashed n-grams + softmax) |
| B2.1 | `B2.1_bge-m3_svc` | BGE-M3 (568 M) + LinearSVC + Calibrated |
| B2.5 | `B2.5_e5-small_svc` | multilingual-e5-small (118 M) + LinearSVC + Calibrated |

## Метрики

- `macro_f1` / `balanced_accuracy` / `accuracy` — общее качество с учётом
  дисбаланса классов;
- `false_faq_for_anamnesis` — самый опасный класс ошибок (жалоба → FAQ);
- per-class P/R/F1 (включая `anamnesis_recall`) — диагностика по классам;
- `recall_urgent` — safety на `safety_set` (87 urgent-кейсов, все gold=anamnesis);
- `route_accuracy` — на `switch_test` (text-only стресс смены интента);
- `ECE` / `Brier OvR` — калибровка (B1.1 / B2.1);
- `total_ms_per_text` — per-text latency на CPU (encode + predict);
- bootstrap BCa 95% CI + paired bootstrap — статистическая значимость
  (`macro_f1` на test/hard_test, `recall_urgent` на safety_set).

## Архитектура pipeline

Ноутбук является только исполнителем: вся бизнес-логика (split, обучение,
calibration, статистика, графики) живёт в модулях `d1/scripts/*`. Здесь только
переключаются режимы, вызываются готовые функции и отображаются артефакты из
`d1/results/`.

Полный pipeline (`run_d1_pipeline`):

```
run_baselines → benchmark_latency → analyze_confidence →
run_statistical_tests → error_taxonomy → learning_curves → plot_results
```

In [1]:
import os
import sys
import warnings
from pathlib import Path

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings(
    "ignore",
    message=r".*(matmul|overflow|divide by zero).*",
    category=RuntimeWarning,
)

STUDY_ROOT = Path.cwd()
if STUDY_ROOT.name == "d1":
    STUDY_ROOT = STUDY_ROOT.parent
if str(STUDY_ROOT) not in sys.path:
    sys.path.insert(0, str(STUDY_ROOT))

print(f"STUDY_ROOT: {STUDY_ROOT}")
print(f"Python:     {sys.version.split()[0]} ({sys.executable})")


STUDY_ROOT: /Users/kazdoraw/developer/med-agent/study
Python:     3.13.2 (/Users/kazdoraw/developer/med-agent/study/.venv/bin/python)


## 0. Управление экспериментом

**Текущий режим — переиспользование кэша моделей и эмбеддингов; пересчитывается
только постобработка и графики.** Splits/leakage_audit/purge выключены,
`PIPELINE_RERUN=True` запускает все 7 шагов, но обученные модели берутся
из `d1/results/models/`.

Почему это безопасно: cleanup-релиз 2026-05-11 затронул только
постобработку — `eval_metrics.summary_dict`, `statistical_tests.METRICS`,
`learning_curves._LC_METRIC_FILTER`, `plot_results.plot_summary_table` и
документацию. Файлы обучения (`b0_rules.py`, `b1_tfidf.py`,
`b1_fasttext.py`, `b2_embedding.py`) и `train` CSV не менялись → SSoT
`train_bundle.CacheKey` рассчитывает тот же `code_hash` + `dataset_hash`
+ `params_hash`, кэш переиспользуется автоматически.

Ожидаемое время прогона: ~10–12 мин (основной вклад — `learning_curves`,
~10 мин: sub-samples 10/25/50/75/100% × 5 seeds обучаются в TemporaryDirectory
и принципиально не кэшируются). Остальные шаги — секунды.

**Управляющие флаги:**

| Флаг | Что делает | Когда включать | Время |
|---|---|---|---|
| `RESPLIT_DATASET` | Перезапускает `split_dataset.run_split` (text-dedup внутри train и cross-split) | После правок anti-leakage / dataset CSV | ~5 c |
| `RUN_LEAKAGE_AUDIT` | Запускает `leakage_audit.run_audit` (within-train + cross-split + cosine ≥ 0.92) | После RESPLIT, проверить отсутствие утечек | ~30 c |
| `FORCE_CACHE_PURGE` | Удаляет все файлы из `d1/results/models/` | Только если правили код обучения (b0/b1_tfidf/b1_fasttext/b2_embedding.py) и хотите гарантировать свежее обучение | мгновенно |
| `PIPELINE_RERUN` | Полный D1 pipeline: 7 шагов от `run_baselines` до `plot_results` | Всегда после правок в постобработке / графиках | ~10–12 мин (cached) / ~13–15 мин (cold) |

**Точечные флаги** (если pipeline уже отработал и нужна только конкретная часть):

| Флаг | Шаг |
|---|---|
| `RUN_BASELINES` | `run_baselines.run_all_baselines` |
| `RUN_BENCHMARK_LATENCY` | `benchmark_latency.run_benchmark` |
| `RUN_ANALYZE_CONFIDENCE` | `analyze_confidence.run_confidence_analysis` |
| `RUN_STATISTICAL_TESTS` | `run_statistical_tests.run_statistical_tests` |
| `RUN_ERROR_TAXONOMY` | `error_taxonomy.run_error_taxonomy` |
| `RUN_LEARNING_CURVES` | `learning_curves.run_learning_curves` (тяжёлый, ~10 мин) |
| `RUN_PLOT_RESULTS` | `plot_results.run_all_plots` |

**Cache invalidation:** `train_bundle.CacheKey` собирает SHA-256 хеш из 5
компонентов (`params_hash`, `dataset_hash`, `code_hash`, `env_hash`,
`schema_hash`). Если хоть один компонент изменился — соответствующий
baseline переобучается автоматически. `FORCE_CACHE_PURGE=True` нужен
только для гарантии переобучения при битово-идентичных входах (например,
после ручной правки внутренних констант, не вошедшей в `code_hash`).

In [2]:
# === МАСТЕР-ФЛАГИ (текущий режим: «переиспользовать кэш моделей,
#     пересчитать только постобработку и графики») ===
# Splits, эмбеддинги и веса моделей не пересчитываются.
# Кэш в d1/results/models/ переиспользуется (cache_key совпадает,
# т.к. b0_rules/b1_tfidf/b1_fasttext/b2_embedding.py не менялись).
RESPLIT_DATASET        = False
RUN_LEAKAGE_AUDIT      = False
FORCE_CACHE_PURGE      = False
PIPELINE_RERUN         = True

# === Точечные флаги ===
# Все шаги pipeline выполняются через PIPELINE_RERUN. Точечные флаги
# нужны, если требуется прогнать только конкретный шаг (например, после
# правок в plot_results.py — RUN_PLOT_RESULTS=True, PIPELINE_RERUN=False).
RUN_BASELINES          = False
RUN_BENCHMARK_LATENCY  = False
RUN_ANALYZE_CONFIDENCE = False
RUN_STATISTICAL_TESTS  = False
RUN_ERROR_TAXONOMY     = False
RUN_LEARNING_CURVES    = False
RUN_PLOT_RESULTS       = False

_master = {
    "RESPLIT_DATASET":   RESPLIT_DATASET,
    "RUN_LEAKAGE_AUDIT": RUN_LEAKAGE_AUDIT,
    "FORCE_CACHE_PURGE": FORCE_CACHE_PURGE,
    "PIPELINE_RERUN":    PIPELINE_RERUN,
}
_targeted = {
    "RUN_BASELINES":          RUN_BASELINES,
    "RUN_BENCHMARK_LATENCY":  RUN_BENCHMARK_LATENCY,
    "RUN_ANALYZE_CONFIDENCE": RUN_ANALYZE_CONFIDENCE,
    "RUN_STATISTICAL_TESTS":  RUN_STATISTICAL_TESTS,
    "RUN_ERROR_TAXONOMY":     RUN_ERROR_TAXONOMY,
    "RUN_LEARNING_CURVES":    RUN_LEARNING_CURVES,
    "RUN_PLOT_RESULTS":       RUN_PLOT_RESULTS,
}

print("=== Мастер-флаги ===")
for name, val in _master.items():
    print(f"  [{'ON ' if val else 'off'}] {name}")
print("\n=== Точечные флаги ===")
for name, val in _targeted.items():
    print(f"  [{'ON ' if val else 'off'}] {name}")

if not any({**_master, **_targeted}.values()):
    print("\nВсе флаги выключены — ноутбук в режиме чтения d1/results/.")

=== Мастер-флаги ===
  [off] RESPLIT_DATASET
  [off] RUN_LEAKAGE_AUDIT
  [off] FORCE_CACHE_PURGE
  [ON ] PIPELINE_RERUN

=== Точечные флаги ===
  [off] RUN_BASELINES
  [off] RUN_BENCHMARK_LATENCY
  [off] RUN_ANALYZE_CONFIDENCE
  [off] RUN_STATISTICAL_TESTS
  [off] RUN_ERROR_TAXONOMY
  [off] RUN_LEARNING_CURVES
  [off] RUN_PLOT_RESULTS


## 1. Данные: пересборка splits + leakage-аудит

`d1/scripts/split_dataset.py` после cleanup-релиза 2026-05-11 делает:

1. **Group-aware split по `seed_id`** (вариации одного семантического зерна
   → один split) — защита от семейного leakage.
2. **Text-level dedup внутри train** — удаление точных дубликатов текста,
   которые LLM-аугментация может породить из разных seed-семей.
3. **Text-level dedup cross-split** — между disjoint primary splits в фиксированном
   приоритете: train > val > test > hard_test > blind_test > switch_test > extended_eval.
4. **Refresh subset-views** — `safety_set` (⊂ hard_test) и `entity_held_out`
   (⊂ test) пересобираются после dedup родителей.

Затем `leakage_audit` проверяет: within-train dups, seed-overlap, exact-text
duplicates, cosine ≥ 0.92. Семантические парафразы (cos ≥ 0.92) — это
legitimate noise (в production пользователи тоже пишут парафразы), не
блокирует pipeline.

In [3]:
import pandas as pd

from d1.config import DATA_DIR, DATASET_PREFIX

if RESPLIT_DATASET:
    from d1.scripts.split_dataset import run_split
    print("Пересборка splits...")
    run_split()
    print()

# Сводка по splits.
split_names = [
    "train", "val", "test", "hard_test", "safety_set",
    "blind_test", "entity_held_out", "extended_eval", "switch_test",
]
rows = []
for name in split_names:
    path = DATA_DIR / f"{DATASET_PREFIX}_{name}.csv"
    if not path.exists():
        rows.append({"split": name, "n": 0, "anamnesis": 0, "faq": 0, "booking": 0, "unsupported": 0})
        continue
    df = pd.read_csv(path)
    dist = df["route_domain"].value_counts().to_dict() if "route_domain" in df.columns else {}
    rows.append({
        "split": name,
        "n": len(df),
        "anamnesis": int(dist.get("anamnesis", 0)),
        "faq": int(dist.get("faq", 0)),
        "booking": int(dist.get("booking", 0)),
        "unsupported": int(dist.get("unsupported", 0)),
    })

print("=== Размеры и распределение классов ===")
display(pd.DataFrame(rows))


=== Размеры и распределение классов ===


,split,n,anamnesis,faq,booking,unsupported
0,train,2140,671,670,514,285
1,val,330,95,110,55,70
2,test,709,215,253,153,88
3,hard_test,204,154,27,17,6
4,safety_set,87,87,0,0,0
5,blind_test,38,12,12,8,6
6,entity_held_out,100,29,61,10,0
7,extended_eval,101,0,101,0,0
8,switch_test,38,15,13,10,0


In [4]:
if RUN_LEAKAGE_AUDIT:
    from d1.scripts.leakage_audit import run_audit
    print("Запуск leakage-аудита...\n")
    results = run_audit()
    print("\n=== Сводка ===")
    for k, v in results.items():
        status = "OK" if v == 0 else "ISSUES"
        print(f"  [{status:>6}] {k}: {v}")
else:
    print("Leakage audit пропущен (RUN_LEAKAGE_AUDIT=False).")

Leakage audit пропущен (RUN_LEAKAGE_AUDIT=False).


## 2. Полный D1 pipeline

Запускает последовательно 7 шагов через `subprocess.run(sys.executable, ...)`,
чтобы гарантировать запуск в текущем venv (см. `d1.scripts.notebook_runner.PIPELINE_STEPS`):

```
run_baselines        — train + closed-set eval на 6 standard eval-сетах +
                       safety_results + switch_results (5 baseline)
benchmark_latency    — per-text encode/predict (n=100, repeats=5)
analyze_confidence   — calibration B1.1 / B2.1: ECE, Brier, reliability tables,
                       confidence_dist (test + hard_test, без val)
run_statistical_tests — bootstrap BCa CI + paired bootstrap. macro_f1+recall_anam
                       на test/hard_test; recall_urgent — только на safety_set
error_taxonomy       — multi-label категоризация ошибок B1.1 ∪ B2.1
learning_curves      — sample-efficiency: B1.1/B2.1 × 5 fractions × 5 seeds × 3 eval (~10 мин)
plot_results         — все PNG (routing/safety/learning_curves/...)
```

При `FORCE_CACHE_PURGE=True` перед запуском удаляются файлы из
`d1/results/models/`, что гарантирует свежее обучение всех 5 baseline'ов.

In [5]:
import shutil

from d1.config import RESULTS_DIR
from d1.baselines.trained_bundle import MODELS_DIR

if FORCE_CACHE_PURGE:
    if MODELS_DIR.exists():
        print(f"Удаление кэша моделей: {MODELS_DIR}")
        for item in MODELS_DIR.iterdir():
            if item.is_dir():
                shutil.rmtree(item)
            else:
                item.unlink()
        print("  → кэш очищен (все модели будут обучены заново)")
    else:
        print(f"Директория {MODELS_DIR} ещё не создана — пропуск")
    print()

if PIPELINE_RERUN:
    from d1.scripts.notebook_runner import run_d1_pipeline
    print("Запуск полного D1 pipeline (7 шагов)...")
    print("  run_baselines → benchmark_latency → analyze_confidence →")
    print("  run_statistical_tests → error_taxonomy → learning_curves → plot_results\n")
    run_d1_pipeline()
    print("\n✓ Pipeline завершён. Артефакты в d1/results/.")
else:
    print("PIPELINE_RERUN=False — ноутбук читает существующие артефакты из d1/results/.")

Запуск полного D1 pipeline (7 шагов)...
  run_baselines → benchmark_latency → analyze_confidence →
  run_statistical_tests → error_taxonomy → learning_curves → plot_results

D1 pipeline: 7 шагов
Интерпретатор: /Users/kazdoraw/developer/med-agent/study/.venv/bin/python
Рабочая директория: /Users/kazdoraw/developer/med-agent/study

[1/7] d1.scripts.run_baselines


2026-05-11 13:27:53,438 [INFO] __main__: === train_bundle: 5 enabled baselines ===
2026-05-11 13:27:53,443 [INFO] d1.baselines.trained_bundle: train_bundle: 5 baseline'ов, train=2140 строк
2026-05-11 13:27:55,881 [INFO] d1.baselines.trained_bundle: cache_hit: B0_rules ← b0_rules.joblib
2026-05-11 13:27:55,907 [INFO] d1.baselines.trained_bundle: cache_hit: B1.1_tfidf_lr ← b1_1_tfidf_lr.joblib
2026-05-11 13:27:55,909 [INFO] d1.baselines.trained_bundle: cache_hit: B2.1_bge-m3_svc ← b2_1_bge_m3_svc.joblib
2026-05-11 13:27:55,910 [INFO] d1.baselines.trained_bundle: cache_hit: B2.5_e5-small_svc ← b2_5_e5_small_svc.joblib
2026-05-11 13:27:56,245 [INFO] d1.baselines.trained_bundle: cache_hit: B1.3_fasttext ← b1_3_fasttext_ft/
2026-05-11 13:27:56,245 [INFO] __main__: 
--- Evaluating on: test ---
2026-05-11 13:27:56,247 [INFO] __main__: Loaded test: 709 rows
2026-05-11 13:27:56,378 [INFO] sentence_transformers.SentenceTransformer: Use pytorch device_name: mps
2026-05-11 13:27:56,378 [INFO] sente


  B0_rules @ test
  Samples:           709
  Accuracy:          0.3907
  Macro-F1:          0.4213
  Balanced Accuracy: 0.4819
  Latency:           0.02 ms/req

  Safety:
    recall(anamnesis):         0.2977
    false_faq_for_anamnesis:   0.0047
    recall(urgent/emergency):  0.9412

  Per-class:
    anamnesis        P=0.571  R=0.298  F1=0.391  n=215
    faq              P=0.985  R=0.257  F1=0.408  n=253
    booking          P=0.955  R=0.418  F1=0.582  n=153
    unsupported      P=0.181  R=0.955  F1=0.304  n=88


  B1.1_tfidf_lr @ test
  Samples:           709
  Accuracy:          0.8632
  Macro-F1:          0.8398
  Balanced Accuracy: 0.8217
  Latency:           0.03 ms/req

  Safety:
    recall(anamnesis):         0.8837
    false_faq_for_anamnesis:   0.1023
    recall(urgent/emergency):  0.4118

  Per-class:
    anamnesis        P=0.848  R=0.884  F1=0.866  n=215
    faq              P=0.821  R=0.905  F1=0.861  n=253
    booking          P=0.954  R=0.941  F1=0.947  n=153
    unsupp

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 66665.56it/s]



  B2.1_bge-m3_svc @ test
  Samples:           709
  Accuracy:          0.8858
  Macro-F1:          0.8777
  Balanced Accuracy: 0.8732
  Latency:           2.89 ms/req

  Safety:
    recall(anamnesis):         0.8651
    false_faq_for_anamnesis:   0.0977
    recall(urgent/emergency):  0.4118

  Per-class:
    anamnesis        P=0.949  R=0.865  F1=0.905  n=215
    faq              P=0.858  R=0.909  F1=0.883  n=253
    booking          P=0.861  R=0.935  F1=0.897  n=153
    unsupported      P=0.873  R=0.784  F1=0.826  n=88



...

2026-05-11 13:32:04,408 [INFO] __main__: Learning curve: fraction=0.10 seed=11 rows=206 families=27
2026-05-11 13:32:04,410 [INFO] d1.baselines.trained_bundle: train_bundle: 2 baseline'ов, train=206 строк
2026-05-11 13:32:05,726 [INFO] d1.baselines.trained_bundle: Обучение B1.1_tfidf_lr (params={'head_type': 'logistic'})
2026-05-11 13:32:05,744 [INFO] d1.baselines.trained_bundle: Обучение B1.1_tfidf_lr готово за 17.8 мс
2026-05-11 13:32:05,826 [INFO] d1.baselines.trained_bundle: Обучение B2.1_bge-m3_svc (params={'head_type': 'svc'})
2026-05-11 13:32:05,826 [INFO] sentence_transformers.SentenceTransformer: Load pretrained SentenceTransformer: /Users/kazdoraw/.cache/huggingface/hub/models--BAAI--bge-m3/snapshots/5617a9f61b028005a4858fdac845db406aefb181
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 66997.83it/s]
2026-05-11 13:32:08,399 [INFO] d1.baselines.trained_bundle: Обучение B2.1_bge-m3_svc готово за 2573.4 мс
2026-05-11 13:32:14,800 [INFO] __main__: Learning curve: fractio

[7/7] d1.scripts.plot_results

=== ROUTING SUMMARY (test) ===
         Baseline  Accuracy  Macro-F1  Bal.Acc  FAQ-for-anam  Latency, ms/text (n=100)
         B0 Rules    0.3907    0.4213   0.4819        0.0047                    0.0278
   B1.1 TF-IDF+LR    0.8632    0.8398   0.8217        0.1023                    0.0262
    B1.3 fastText    0.8265    0.8008   0.7964        0.0977                    0.0032
     B2.1 BGE+SVC    0.8858    0.8777   0.8732        0.0977                    7.4607
B2.5 E5-small+SVC    0.8533    0.8446   0.8450        0.0977                    1.0600


2026-05-11 13:38:12,384 [INFO] __main__: Saved: /Users/kazdoraw/developer/med-agent/study/d1/results/figures/routing_comparison_test.png
2026-05-11 13:38:12,434 [INFO] __main__: Saved: /Users/kazdoraw/developer/med-agent/study/d1/results/figures/routing_comparison_hard_test.png
2026-05-11 13:38:12,700 [INFO] __main__: Saved: /Users/kazdoraw/developer/med-agent/study/d1/results/figures/confusion_matrices_test.png
2026-05-11 13:38:12,969 [INFO] __main__: Saved: /Users/kazdoraw/developer/med-agent/study/d1/results/figures/confusion_matrices_hard_test.png
2026-05-11 13:38:13,028 [INFO] __main__: Saved: /Users/kazdoraw/developer/med-agent/study/d1/results/figures/per_class_f1_test.png
2026-05-11 13:38:13,084 [INFO] __main__: Saved: /Users/kazdoraw/developer/med-agent/study/d1/results/figures/per_class_f1_hard_test.png
2026-05-11 13:38:13,160 [INFO] __main__: Saved: /Users/kazdoraw/developer/med-agent/study/d1/results/figures/safety_comparison.png
2026-05-11 13:38:13,205 [INFO] __main__: Sav


✓ Сохранено 11 артефактов в /Users/kazdoraw/developer/med-agent/study/d1/results/figures

✓ D1 pipeline complete. Артефакты: /Users/kazdoraw/developer/med-agent/study/d1/results

✓ Pipeline завершён. Артефакты в d1/results/.


2026-05-11 13:38:13,425 [INFO] __main__: Saved: /Users/kazdoraw/developer/med-agent/study/d1/results/figures/cross_eval_f1.png


## 3. Состояние моделей в кэше

Показывает 5 канонических моделей, размеры артефактов, время обучения и
cache-keys (`params_hash`, `dataset_hash`, `code_hash`, `env_hash`,
`schema_hash`).

In [6]:
import json

from d1.baselines.trained_bundle import BASELINE_CONFIGS, MODELS_DIR

metadata_path = MODELS_DIR / "bundle_metadata.json"
print(f"Реестр BASELINE_CONFIGS: {len(BASELINE_CONFIGS)} моделей")
print(f"Директория кэша:        {MODELS_DIR}\n")

if not metadata_path.exists():
    print("В кэше нет ни одной модели — запустите PIPELINE_RERUN=True.")
else:
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    print(f"{'Модель':22s} {'Размер':>10s}  {'Обучено':19s}  {'fit_ms':>10s}  train_size")
    print("-" * 85)
    for name in BASELINE_CONFIGS:
        slug = name.lower().replace(".", "_").replace("-", "_")
        joblib_path = MODELS_DIR / f"{slug}.joblib"
        ft_path = MODELS_DIR / f"{slug}_ft"
        if joblib_path.exists():
            size_str = f"{joblib_path.stat().st_size / 1024:>8.1f} KB"
        elif ft_path.exists():
            total = sum(p.stat().st_size for p in ft_path.rglob('*') if p.is_file())
            size_str = f"{total / 1024:>8.1f} KB"
        else:
            size_str = "—"
        meta = metadata.get(name, {})
        saved = meta.get("saved_at", "—")[:19].replace("T", " ")
        fit_ms = meta.get("fit_time_ms")
        fit_str = f"{fit_ms:>10.1f}" if isinstance(fit_ms, (int, float)) else f"{'—':>10}"
        train_size = meta.get("train_size", "—")
        print(f"{name:22s} {size_str}  {saved:19s}  {fit_str}  {train_size}")

    print("\n=== Cache keys (для первой модели в качестве отпечатка прогона) ===")
    first = next(iter(BASELINE_CONFIGS))
    if first in metadata and "cache_key" in metadata[first]:
        for k, v in metadata[first]["cache_key"].items():
            print(f"  {k:14s} = {v}")

Реестр BASELINE_CONFIGS: 5 моделей
Директория кэша:        /Users/kazdoraw/developer/med-agent/study/d1/results/models

Модель                     Размер  Обучено                  fit_ms  train_size
-------------------------------------------------------------------------------------
B0_rules                    0.1 KB  —                             —  —
B1.1_tfidf_lr             846.3 KB  2026-05-11 09:28:34       180.3  2140
B2.1_bge-m3_svc           165.9 KB  2026-05-11 09:28:42      7841.3  2140
B2.5_e5-small_svc          65.9 KB  —                             —  —
B1.3_fasttext          782395.9 KB  —                             —  —

=== Cache keys (для первой модели в качестве отпечатка прогона) ===


## 4. Закрытая классификация: сводка по eval-сетам

Каждая модель возвращает ровно одну метку для каждого сообщения (closed-set
top-1). Считаются `accuracy`, `macro_F1`, `balanced_accuracy`,
`false_faq_for_anamnesis` и per-class P/R/F1 (включая `anamnesis_recall`
как диагностику безопасности по классу) для всех 5 baseline на 6 standard
eval-сетах.

In [7]:
import pandas as pd

from d1.config import RESULTS_DIR

baseline_csv = RESULTS_DIR / "baseline_results.csv"
if not baseline_csv.exists():
    print("baseline_results.csv ещё не создан — запустите PIPELINE_RERUN=True.")
else:
    df = pd.read_csv(baseline_csv)
    df["baseline"] = df["baseline"].str.split(" @ ").str[0]
    cols = [
        "baseline", "eval_set", "accuracy", "macro_f1", "balanced_accuracy",
        "anamnesis_recall", "false_faq_for_anamnesis",
    ]
    df = df[cols].round(4)

    for ev in ["test", "hard_test", "blind_test", "entity_held_out", "extended_eval", "val"]:
        sub = df[df["eval_set"] == ev].sort_values("macro_f1", ascending=False)
        if sub.empty:
            continue
        print(f"\n=== {ev} (n={int(sub.shape[0])} моделей) ===")
        display(sub.drop(columns=["eval_set"]).reset_index(drop=True))

    ml = df[df["baseline"] != "B0_rules"]
    print("\n=== Разброс (max − min) по 4 ML на test/hard_test ===")
    spread_rows = []
    for ev in ["test", "hard_test"]:
        sub = ml[ml["eval_set"] == ev]
        for m in ["accuracy", "macro_f1", "balanced_accuracy", "anamnesis_recall"]:
            v = sub[m].values
            if len(v) >= 2:
                spread_rows.append({
                    "eval_set": ev, "metric": m,
                    "min": round(v.min(), 4), "max": round(v.max(), 4),
                    "spread": round(v.max() - v.min(), 4),
                })
    display(pd.DataFrame(spread_rows))


=== test (n=5 моделей) ===


,baseline,accuracy,macro_f1,balanced_accuracy,anamnesis_recall,false_faq_for_anamnesis
0,B2.1_bge-m3_svc,0.8858,0.8777,0.8732,0.8651,0.0977
1,B2.5_e5-small_svc,0.8533,0.8446,0.8450,0.8698,0.0977
2,B1.1_tfidf_lr,0.8632,0.8398,0.8217,0.8837,0.1023
3,B1.3_fasttext,0.8265,0.8008,0.7964,0.8651,0.0977
4,B0_rules,0.3907,0.4213,0.4819,0.2977,0.0047



=== hard_test (n=5 моделей) ===


,baseline,accuracy,macro_f1,balanced_accuracy,anamnesis_recall,false_faq_for_anamnesis
0,B1.1_tfidf_lr,0.7353,0.6102,0.6744,0.7597,0.1623
1,B2.1_bge-m3_svc,0.7108,0.5982,0.7238,0.7013,0.2078
2,B1.3_fasttext,0.6961,0.5850,0.6898,0.6883,0.2013
3,B2.5_e5-small_svc,0.7108,0.5745,0.7071,0.7338,0.1948
4,B0_rules,0.6569,0.4652,0.6192,0.7403,0.0325



=== blind_test (n=5 моделей) ===


...

,baseline,accuracy,macro_f1,balanced_accuracy,anamnesis_recall,false_faq_for_anamnesis
0,B2.1_bge-m3_svc,0.7822,0.2194,0.7822,0.0,0.0
1,B1.1_tfidf_lr,0.7426,0.2131,0.7426,0.0,0.0
2,B2.5_e5-small_svc,0.7228,0.2098,0.7228,0.0,0.0
3,B1.3_fasttext,0.7030,0.2064,0.7030,0.0,0.0
4,B0_rules,0.0891,0.0409,0.0891,0.0,0.0



=== val (n=5 моделей) ===


,baseline,accuracy,macro_f1,balanced_accuracy,anamnesis_recall,false_faq_for_anamnesis
0,B2.1_bge-m3_svc,0.8667,0.8600,0.8649,0.9895,0.0105
1,B1.1_tfidf_lr,0.8182,0.8016,0.8042,0.9053,0.0842
2,B2.5_e5-small_svc,0.8030,0.7886,0.7998,0.9263,0.0526
3,B1.3_fasttext,0.7848,0.7619,0.7691,0.8737,0.0421
4,B0_rules,0.4818,0.4881,0.4988,0.4211,0.0000



=== Разброс (max − min) по 4 ML на test/hard_test ===


,eval_set,metric,min,max,spread
0,test,accuracy,0.8265,0.8858,0.0593
1,test,macro_f1,0.8008,0.8777,0.0769
2,test,balanced_accuracy,0.7964,0.8732,0.0768
3,test,anamnesis_recall,0.8651,0.8837,0.0186
4,hard_test,accuracy,0.6961,0.7353,0.0392
5,hard_test,macro_f1,0.5745,0.6102,0.0357
6,hard_test,balanced_accuracy,0.6744,0.7238,0.0494
7,hard_test,anamnesis_recall,0.6883,0.7597,0.0714


## 5. Safety: recall_urgent (safety_set, n=87)

Все 87 сообщений — urgent/emergency с gold=anamnesis. Главная safety-метрика.
Артефакт: `d1/results/safety_results.{csv,json}`.

In [8]:
import pandas as pd

from d1.config import RESULTS_DIR

safety_csv = RESULTS_DIR / "safety_results.csv"
if not safety_csv.exists():
    print("safety_results.csv ещё не создан.")
else:
    df = pd.read_csv(safety_csv)
    df["baseline"] = df["baseline"].str.split(" @ ").str[0]
    df = df.sort_values("recall_urgent", ascending=False).round(4)
    display(df[["baseline", "n_samples", "recall_urgent", "false_negative_urgent",
                "false_faq_for_anamnesis", "misrouted_to"]])

    # Подсветим лидеров.
    print("\nЛидеры по recall_urgent:")
    top = df[df["recall_urgent"] == df["recall_urgent"].max()]["baseline"].tolist()
    print(f"  {', '.join(top)} = {df['recall_urgent'].max():.4f}")

,baseline,n_samples,recall_urgent,false_negative_urgent,false_faq_for_anamnesis,misrouted_to
3,B2.1_bge-m3_svc,87,0.9310,6,0.0345,"{'booking': 2, 'faq': 3, 'unsupported': 1}"
1,B1.1_tfidf_lr,87,0.9195,7,0.0575,"{'faq': 5, 'booking': 1, 'unsupported': 1}"
4,B2.5_e5-small_svc,87,0.9195,7,0.0345,"{'faq': 3, 'unsupported': 2, 'booking': 2}"
2,B1.3_fasttext,87,0.8506,13,0.0575,"{'booking': 6, 'unsupported': 2, 'faq': 5}"
0,B0_rules,87,0.7471,22,0.0000,{'unsupported': 22}



Лидеры по recall_urgent:
  B2.1_bge-m3_svc = 0.9310


## 6. Switch test: смена интента в диалоге (n=50)

Модель видит **только** текст следующей реплики; `active_domain` предыдущего
диалога используется лишь для per-transition breakdown в отчёте. Артефакт:
`d1/results/switch_results.{csv,json}`.

In [9]:
import pandas as pd

from d1.config import RESULTS_DIR

switch_csv = RESULTS_DIR / "switch_results.csv"
if not switch_csv.exists():
    print("switch_results.csv ещё не создан.")
else:
    df = pd.read_csv(switch_csv).round(4)
    df["baseline"] = df["baseline"].str.split(" @ ").str[0]
    cols = [c for c in df.columns if c not in ("eval_set", "latency_ms")]
    display(df[cols].sort_values("route_accuracy", ascending=False).reset_index(drop=True))

,baseline,n_samples,route_accuracy,acc__anamnesis->booking,acc__anamnesis->faq,acc__booking->anamnesis,acc__booking->faq,acc__faq->anamnesis,acc__faq->booking
0,B2.1_bge-m3_svc,38,0.9474,1.0,1.0000,0.8889,0.8333,1.0000,1.0000
1,B1.1_tfidf_lr,38,0.9211,1.0,0.8571,0.8889,0.8333,1.0000,1.0000
2,B1.3_fasttext,38,0.8947,1.0,1.0000,0.6667,1.0000,1.0000,0.8333
3,B2.5_e5-small_svc,38,0.8947,1.0,0.8571,0.8889,0.8333,1.0000,0.8333
4,B0_rules,38,0.7105,1.0,0.4286,0.6667,0.6667,0.8333,0.8333


## 7. Латентность на CPU (`latency_breakdown.csv`)

Per-text median + p95 на 100 текстах × 5 повторов. Разделение на `encode_ms`
и `predict_ms` показывает, у каких моделей энкодер доминирует.

In [10]:
import pandas as pd

from d1.config import RESULTS_DIR

if RUN_BENCHMARK_LATENCY:
    from d1.scripts.benchmark_latency import run_benchmark
    print("Замер латентности (n=100, repeats=5)...")
    run_benchmark(n=100, repeats=5)

latency_csv = RESULTS_DIR / "latency_breakdown.csv"
if not latency_csv.exists():
    print("latency_breakdown.csv ещё не создан.")
else:
    df = pd.read_csv(latency_csv).round(4)
    df = df.sort_values("total_ms_per_text_median")
    cols = [
        "baseline", "total_ms_per_text_median",
        "encode_ms_per_text_median", "predict_ms_per_text_median",
        "total_ms_p95", "encode_share",
    ]
    display(df[[c for c in cols if c in df.columns]].reset_index(drop=True))

,baseline,total_ms_per_text_median,encode_ms_per_text_median,predict_ms_per_text_median,total_ms_p95,encode_share
0,B1.3_fasttext,0.0032,0.0000,0.0032,0.338,0.000
1,B1.1_tfidf_lr,0.0262,0.0251,0.0010,2.732,0.957
2,B0_rules,0.0278,0.0000,0.0278,2.778,0.000
3,B2.5_e5-small_svc,1.0600,1.0489,0.0099,115.068,0.989
4,B2.1_bge-m3_svc,7.4607,7.4496,0.0112,770.112,0.999


## 8. Калибровка confidence (B1.1 / B2.1)

ECE и Brier OvR macro для двух моделей с осмысленной `predict_proba`. Артефакты:
`calibration_metrics_*.json`, `reliability_table_*.csv`. Считаются только на
`test` и `hard_test` (без `val` — он используется только для tuning).

Cleanup-релиз 2026-05-11: `threshold_table_*.csv` и `threshold_curve_*.png`
удалены — это были relic selective/hybrid эпохи (coverage vs accepted accuracy
для defer-логики), не нужны в чистом closed-set top-1.

In [11]:
import glob
import json

import pandas as pd

from d1.config import RESULTS_DIR

if RUN_ANALYZE_CONFIDENCE:
    from d1.scripts.analyze_confidence import run_confidence_analysis
    print("Запуск analyze_confidence...")
    run_confidence_analysis()

paths = sorted(glob.glob(str(RESULTS_DIR / "calibration_metrics_*.json")))
if not paths:
    print("calibration_metrics_*.json не найдены.")
else:
    rows = []
    for p in paths:
        d = json.loads(open(p, encoding="utf-8").read())
        rows.append({
            "baseline": d.get("baseline"),
            "eval_set": d.get("eval_set"),
            "ECE": round(d.get("ece", float("nan")), 4),
            "Brier(macro)": round(d.get("brier_ovr", {}).get("macro", float("nan")), 4),
        })
    display(pd.DataFrame(rows).sort_values(["eval_set", "baseline"]).reset_index(drop=True))

,baseline,eval_set,ECE,Brier(macro)
0,B1.1 TF-IDF+LR,hard_test,0.0933,0.1009
1,B2.1 BGE-M3+SVC,hard_test,0.1144,0.1078
2,B1.1 TF-IDF+LR,test,0.2111,0.0686
3,B2.1 BGE-M3+SVC,test,0.0712,0.0428


## 9. Статистическая значимость

- **Bootstrap BCa 95% CI**: для 5 baseline. `macro_f1` на `test` + `hard_test`;
  `recall_urgent` на `safety_set`.
- **Paired bootstrap**: 10 пар (C(5,2)), one-sided p-value + колонка
  `significant` (CI не пересекает 0).
- `recall_anamnesis` убран из статистических тестов: для сравнения моделей
  достаточно `macro_f1` (агрегатное качество) + `recall_urgent` на
  специально сконструированном safety_set. Per-class `anamnesis_recall`
  остаётся как диагностика в `baseline_results.json`.

In [12]:
import pandas as pd

from d1.config import RESULTS_DIR

if RUN_STATISTICAL_TESTS:
    from d1.scripts.run_statistical_tests import run_statistical_tests
    print("Запуск run_statistical_tests...")
    run_statistical_tests()

ci_csv = RESULTS_DIR / "bootstrap_ci.csv"
pt_csv = RESULTS_DIR / "paired_tests.csv"

if ci_csv.exists():
    ci = pd.read_csv(ci_csv).round(4)
    for ev, metric in [("test", "macro_f1"), ("hard_test", "macro_f1"), ("safety_set", "recall_urgent")]:
        show = ci[(ci["eval_set"] == ev) & (ci["metric"] == metric)]
        if show.empty:
            continue
        print(f"\n=== Bootstrap BCa 95% CI ({ev}, {metric}) ===")
        display(show[["baseline", "metric", "point", "ci_lower", "ci_upper"]].reset_index(drop=True))
else:
    print("bootstrap_ci.csv не найден.")

if pt_csv.exists():
    pt = pd.read_csv(pt_csv).round(4)
    cols = ["baseline_a", "baseline_b", "delta_mean",
            "delta_ci_low", "delta_ci_high", "p_value_one_sided"]
    if "significant" in pt.columns:
        cols.append("significant")
    for ev, metric in [("test", "macro_f1"), ("hard_test", "macro_f1"), ("safety_set", "recall_urgent")]:
        show = pt[(pt["eval_set"] == ev) & (pt["metric"] == metric)]
        if show.empty:
            continue
        print(f"\n=== Paired bootstrap ({ev}, {metric}, 10 пар) ===")
        display(show[cols].reset_index(drop=True))
else:
    print("paired_tests.csv не найден.")


=== Bootstrap BCa 95% CI (test, macro_f1) ===


,baseline,metric,point,ci_lower,ci_upper
0,B0_rules,macro_f1,0.4213,0.3565,0.4930
1,B1.1_tfidf_lr,macro_f1,0.8398,0.7948,0.8793
2,B1.3_fasttext,macro_f1,0.8008,0.7621,0.8373
3,B2.1_bge-m3_svc,macro_f1,0.8777,0.8315,0.9150
4,B2.5_e5-small_svc,macro_f1,0.8446,0.7963,0.8865



=== Bootstrap BCa 95% CI (hard_test, macro_f1) ===


,baseline,metric,point,ci_lower,ci_upper
0,B0_rules,macro_f1,0.4652,0.3907,0.5558
1,B1.1_tfidf_lr,macro_f1,0.6102,0.4975,0.7303
2,B1.3_fasttext,macro_f1,0.5850,0.4760,0.6959
3,B2.1_bge-m3_svc,macro_f1,0.5982,0.4965,0.7068
4,B2.5_e5-small_svc,macro_f1,0.5745,0.4853,0.6808



=== Bootstrap BCa 95% CI (safety_set, recall_urgent) ===


...

,baseline_a,baseline_b,delta_mean,delta_ci_low,delta_ci_high,p_value_one_sided,significant
0,B0_rules,B1.1_tfidf_lr,-0.4185,-0.4908,-0.3384,1.0000,True
1,B0_rules,B1.3_fasttext,-0.3795,-0.4454,-0.3069,1.0000,True
2,B0_rules,B2.1_bge-m3_svc,-0.4564,-0.5271,-0.3811,1.0000,True
3,B0_rules,B2.5_e5-small_svc,-0.4233,-0.4879,-0.3539,1.0000,True
4,B1.1_tfidf_lr,B1.3_fasttext,0.0390,-0.0018,0.0820,0.0310,False
5,B1.1_tfidf_lr,B2.1_bge-m3_svc,-0.0379,-0.0740,0.0032,0.9655,False
6,B1.1_tfidf_lr,B2.5_e5-small_svc,-0.0048,-0.0459,0.0441,0.5630,False
7,B1.3_fasttext,B2.1_bge-m3_svc,-0.0769,-0.1187,-0.0357,1.0000,True
8,B1.3_fasttext,B2.5_e5-small_svc,-0.0438,-0.0913,0.0025,0.9665,False
9,B2.1_bge-m3_svc,B2.5_e5-small_svc,0.0331,-0.0085,0.0778,0.0575,False



=== Paired bootstrap (hard_test, macro_f1, 10 пар) ===


,baseline_a,baseline_b,delta_mean,delta_ci_low,delta_ci_high,p_value_one_sided,significant
0,B0_rules,B1.1_tfidf_lr,-0.1450,-0.2771,-0.0066,0.9780,True
1,B0_rules,B1.3_fasttext,-0.1198,-0.2375,0.0127,0.9635,False
2,B0_rules,B2.1_bge-m3_svc,-0.1330,-0.2450,-0.0105,0.9795,True
3,B0_rules,B2.5_e5-small_svc,-0.1092,-0.2246,0.0156,0.9565,False
4,B1.1_tfidf_lr,B1.3_fasttext,0.0253,-0.0401,0.0936,0.2345,False
5,B1.1_tfidf_lr,B2.1_bge-m3_svc,0.0120,-0.0894,0.1091,0.4155,False
6,B1.1_tfidf_lr,B2.5_e5-small_svc,0.0358,-0.0804,0.1494,0.2835,False
7,B1.3_fasttext,B2.1_bge-m3_svc,-0.0132,-0.1112,0.0756,0.6145,False
8,B1.3_fasttext,B2.5_e5-small_svc,0.0105,-0.1014,0.1180,0.4450,False
9,B2.1_bge-m3_svc,B2.5_e5-small_svc,0.0238,-0.0615,0.1076,0.3080,False



=== Paired bootstrap (safety_set, recall_urgent, 10 пар) ===


,baseline_a,baseline_b,delta_mean,delta_ci_low,delta_ci_high,p_value_one_sided,significant
0,B0_rules,B1.1_tfidf_lr,-0.1724,-0.2759,-0.0690,0.9995,True
1,B0_rules,B1.3_fasttext,-0.1034,-0.2069,0.0003,0.9750,False
2,B0_rules,B2.1_bge-m3_svc,-0.1839,-0.2989,-0.0805,0.9995,True
3,B0_rules,B2.5_e5-small_svc,-0.1724,-0.2759,-0.0690,0.9985,True
4,B1.1_tfidf_lr,B1.3_fasttext,0.0690,0.0000,0.1494,0.0430,False
5,B1.1_tfidf_lr,B2.1_bge-m3_svc,-0.0115,-0.0690,0.0460,0.7360,False
6,B1.1_tfidf_lr,B2.5_e5-small_svc,0.0000,-0.0690,0.0575,0.5840,False
7,B1.3_fasttext,B2.1_bge-m3_svc,-0.0805,-0.1609,0.0000,0.9860,False
8,B1.3_fasttext,B2.5_e5-small_svc,-0.0690,-0.1494,0.0000,0.9800,False
9,B2.1_bge-m3_svc,B2.5_e5-small_svc,0.0115,-0.0345,0.0690,0.4270,False


## 10. Кривые обучения (sample-efficiency)

B1.1 и B2.1 × {10, 25, 50, 75, 100}% train × 5 random seeds × {`test`,
`hard_test`, `safety_set`}. Артефакты: `learning_curves.csv` (raw) и
`learning_curves_summary.csv` (сводка mean ± std).

In [13]:
import pandas as pd

from d1.config import RESULTS_DIR

if RUN_LEARNING_CURVES:
    from d1.scripts.learning_curves import run_learning_curves
    print("Запуск learning_curves (это займёт ~10 минут)...")
    run_learning_curves()

lc_csv = RESULTS_DIR / "learning_curves_summary.csv"
if not lc_csv.exists():
    print("learning_curves_summary.csv ещё не создан.")
else:
    df = pd.read_csv(lc_csv).round(4)
    df = df[(df["eval_set"] == "test") & (df["metric"] == "macro_f1")]
    df = df.sort_values(["baseline", "fraction"]).reset_index(drop=True)
    print("=== macro_F1 на test по fraction (B1.1 / B2.1) ===")
    display(df[["baseline", "fraction", "mean", "std", "n_runs"]])

=== macro_F1 на test по fraction (B1.1 / B2.1) ===


,baseline,fraction,mean,std,n_runs
0,B1.1_tfidf_lr,0.10,0.5361,0.0635,5
1,B1.1_tfidf_lr,0.25,0.6730,0.0402,5
2,B1.1_tfidf_lr,0.50,0.7623,0.0413,5
3,B1.1_tfidf_lr,0.75,0.8160,0.0167,5
4,B1.1_tfidf_lr,1.00,0.8398,0.0000,5
5,B2.1_bge-m3_svc,0.10,0.7175,0.0545,5
6,B2.1_bge-m3_svc,0.25,0.7984,0.0187,5
7,B2.1_bge-m3_svc,0.50,0.8320,0.0104,5
8,B2.1_bge-m3_svc,0.75,0.8640,0.0081,5
9,B2.1_bge-m3_svc,1.00,0.8777,0.0000,5


## 11. Анализ ошибок (error taxonomy)

Multi-label категоризация ошибок (B1.1 ∪ B2.1) на `test`, `hard_test`,
`blind_test`. Категории: `anamnesis_to_faq`, `anamnesis_to_booking`,
`faq_to_anamnesis`, `mixed_intent_error`, `specialization_confusion`,
`vague_short_error`, `post_treatment_ambiguity`,
`both_wrong`, `models_disagree_both_wrong`.

Cleanup-релиз 2026-05-11: `both_wrong_disagreement` переименован в
`both_wrong` (тег ставился по совпадению ошибок обоих моделей независимо
от того, согласны они между собой). Подмножество с реальным disagreement
B1.1 ≠ B2.1 теперь маркируется отдельным флагом `models_disagree_both_wrong`.

In [14]:
import pandas as pd

from d1.config import RESULTS_DIR

if RUN_ERROR_TAXONOMY:
    from d1.scripts.error_taxonomy import run_error_taxonomy
    print("Запуск error_taxonomy...")
    run_error_taxonomy()

et_csv = RESULTS_DIR / "error_taxonomy_summary.csv"
if not et_csv.exists():
    print("error_taxonomy_summary.csv ещё не создан.")
else:
    df = pd.read_csv(et_csv)
    df = df.sort_values(["eval_set", "count"], ascending=[True, False]).reset_index(drop=True)
    cols = ["eval_set", "error_type", "count", "pct_of_errors", "example_text"]
    df = df[[c for c in cols if c in df.columns]]
    df["pct_of_errors"] = df["pct_of_errors"].round(4)
    if "example_text" in df.columns:
        df["example_text"] = df["example_text"].astype(str).str.slice(0, 60)
    display(df)

,eval_set,error_type,count,pct_of_errors,example_text
0,blind_test,anamnesis_to_faq,2,0.4000,поставили пломбу неделю назад и теперь на горя...
1,blind_test,both_wrong,2,0.4000,поставили пломбу неделю назад и теперь на горя...
2,blind_test,generic_error,2,0.4000,а карту рассрочки прямо у вас оформляют?
3,hard_test,both_wrong,43,0.6143,"У меня болит зуб, сколько стоит лечение?"
4,hard_test,anamnesis_to_faq,25,0.3571,"У меня болит зуб, сколько стоит лечение?"
5,hard_test,generic_error,15,0.2143,"Откололся зуб, больно жевать, запишите срочно"
6,hard_test,vague_short_error,12,0.1714,Лечение
7,hard_test,anamnesis_to_booking,11,0.1571,"Десна кровоточит, можно записаться?"
8,hard_test,models_disagree_both_wrong,8,0.1143,"Десна кровоточит, можно записаться?"
9,hard_test,specialization_confusion,8,0.1143,Хочу к Батылину


## 12. Trade-off карта кандидатов

Сводная таблица: качество × безопасность × латентность × калибровка для
трёх production-кандидатов (B1.1, B2.1, B2.5). На основе значений из
`baseline_results.csv`, `safety_results.csv`, `latency_breakdown.csv`,
`switch_results.csv`, `calibration_metrics_*.json`.

In [15]:
import glob
import json

import pandas as pd

from d1.config import RESULTS_DIR

candidates = ["B1.1_tfidf_lr", "B2.1_bge-m3_svc", "B2.5_e5-small_svc"]

br = pd.read_csv(RESULTS_DIR / "baseline_results.csv") if (RESULTS_DIR / "baseline_results.csv").exists() else None
sr = pd.read_csv(RESULTS_DIR / "safety_results.csv") if (RESULTS_DIR / "safety_results.csv").exists() else None
sw = pd.read_csv(RESULTS_DIR / "switch_results.csv") if (RESULTS_DIR / "switch_results.csv").exists() else None
lat = pd.read_csv(RESULTS_DIR / "latency_breakdown.csv") if (RESULTS_DIR / "latency_breakdown.csv").exists() else None

cal_test = {}
cal_hard = {}
for p in glob.glob(str(RESULTS_DIR / "calibration_metrics_*.json")):
    d = json.loads(open(p, encoding="utf-8").read())
    short = d.get("baseline", "")
    bid = "B1.1_tfidf_lr" if "B1.1" in short else ("B2.1_bge-m3_svc" if "B2.1" in short else None)
    if bid is None:
        continue
    if d.get("eval_set") == "test":
        cal_test[bid] = round(d.get("ece", float("nan")), 4)
    elif d.get("eval_set") == "hard_test":
        cal_hard[bid] = round(d.get("ece", float("nan")), 4)

rows = []
for bid in candidates:
    row = {"candidate": bid}
    if br is not None:
        sub = br[(br["baseline"].str.startswith(bid)) & (br["eval_set"] == "test")]
        row["macro_f1@test"] = round(sub.iloc[0]["macro_f1"], 4) if not sub.empty else None
        sub2 = br[(br["baseline"].str.startswith(bid)) & (br["eval_set"] == "hard_test")]
        row["anam_recall@hard"] = round(sub2.iloc[0]["anamnesis_recall"], 4) if not sub2.empty else None
    if sr is not None:
        sub = sr[sr["baseline"].str.startswith(bid)]
        row["recall_urgent@safety"] = round(sub.iloc[0]["recall_urgent"], 4) if not sub.empty else None
    if sw is not None:
        sub = sw[sw["baseline"].str.startswith(bid)]
        row["route_acc@switch"] = round(sub.iloc[0]["route_accuracy"], 4) if not sub.empty else None
    if lat is not None:
        sub = lat[lat["baseline"] == bid]
        row["latency_ms/text"] = round(sub.iloc[0]["total_ms_per_text_median"], 4) if not sub.empty else None
    row["ECE@test"] = cal_test.get(bid)
    row["ECE@hard"] = cal_hard.get(bid)
    rows.append(row)

display(pd.DataFrame(rows))
print("\nИтоговый отчёт: см. d1/EXPERIMENT_D1.md (§2.3.5 — карта кандидатов).")
print("PNG-графики: d1/results/figures/.")
if RUN_PLOT_RESULTS:
    from d1.scripts.plot_results import run_all_plots
    print("\nПерегенерация графиков...")
    run_all_plots()

,candidate,macro_f1@test,anam_recall@hard,recall_urgent@safety,route_acc@switch,latency_ms/text,ECE@test,ECE@hard
0,B1.1_tfidf_lr,0.8398,0.7597,0.9195,0.9211,0.0262,0.2111,0.0933
1,B2.1_bge-m3_svc,0.8777,0.7013,0.9310,0.9474,7.4607,0.0712,0.1144
2,B2.5_e5-small_svc,0.8446,0.7338,0.9195,0.8947,1.0600,NaN,NaN



Итоговый отчёт: см. d1/EXPERIMENT_D1.md (§2.3.5 — карта кандидатов).
PNG-графики: d1/results/figures/.
